# 1. Dependencies import

In [ ]:
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR != ROOT_DIR.parent and not (
    (ROOT_DIR / "data").exists() and (ROOT_DIR / "logs").exists()
):
    ROOT_DIR = ROOT_DIR.parent

os.chdir(ROOT_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from typing import Any

from logs.cmd.logger_config import setup_logger  # noqa: E402

## 1.2. Constants instances

In [ ]:
existing_logger = logging.getLogger("exploratory_analysis")
for handler in existing_logger.handlers[:]:
    handler.close()
    existing_logger.removeHandler(handler)

logger = setup_logger("exploratory_analysis", save_to_file=True)

SEED = 42
np.random.seed(SEED)
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

RESULTS_DIR = ROOT_DIR / "tests" / "results"
IMAGES_DIR = RESULTS_DIR / "images"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

COLOR_PURPLE_DARK = "#4E31A4"
COLOR_PURPLE_LIGHT = "#D6CFF8"
COLOR_YELLOW_SECONDARY = "#D97706"
PALETTE = [COLOR_PURPLE_DARK, COLOR_YELLOW_SECONDARY]

raw_df = pd.read_csv(ROOT_DIR / "data" / "raw" / "bar_pass_prediction.csv")
pre_train = pd.read_csv(ROOT_DIR / "data" / "processed" / "pre_train.csv")
ref_train = pd.read_csv(ROOT_DIR / "data" / "refined" / "train.csv")

logger.info(f"Raw dataset shape: {raw_df.shape}")
logger.info(f"Processed train (pre_train) shape: {pre_train.shape}")
logger.info(f"Refined train shape: {ref_train.shape}")

# 2. Variables types, Descritive Statics and Anomalies

In [ ]:
num_cols = ["lsat", "ugpa", "zfygpa", "age"]
cat_cols = ["tier", "cluster", "fulltime", "fam_inc", "race1", "gender", "target_risk"]

# numerical descriptive statistics (mean, median, std, min, max, quartiles, IQR outliers)
num_stats_rows = []
for col in num_cols:
    series = pre_train[col].dropna()
    q1 = float(series.quantile(0.25))
    q3 = float(series.quantile(0.75))
    iqr = q3 - q1
    outlier_count = int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())
    num_stats_rows.append(
        {
            "Variable": col,
            "Count": int(series.count()),
            "Mean": round(float(series.mean()), 3),
            "Median": round(float(series.median()), 3),
            "Std": round(float(series.std()), 3),
            "Min": round(float(series.min()), 3),
            "Q1": round(q1, 3),
            "Q3": round(q3, 3),
            "Max": round(float(series.max()), 3),
            "IQR_Outliers": outlier_count,
            "IQR_Outliers_Pct": round(outlier_count / len(series) * 100, 2),
        }
    )

desc_num_df = pd.DataFrame(num_stats_rows)
desc_num_df.to_csv(RESULTS_DIR / "eda_numerical_summary.csv", index=False)
display(desc_num_df)

# categorical variables frequencies and failure rates
cat_freq_rows = []
for col in cat_cols:
    freq = pre_train[col].value_counts(dropna=False)
    pct = (pre_train[col].value_counts(normalize=True, dropna=False) * 100).round(2)
    risk_by_cat = (
        pre_train.groupby(col, dropna=False)["target_risk"].mean() * 100
    ).round(2)

    cat_summary = pd.DataFrame(
        {
            "Count": freq,
            "Percent (%)": pct,
            "Failure_Rate (%)": risk_by_cat,
        }
    )
    logger.info(f"\n--- Category Frequency: {col} ---")
    display(cat_summary)

    for category, row in cat_summary.iterrows():
        is_missing = (
            category is None
            or category is pd.NA
            or (isinstance(category, float) and np.isnan(category))
        )
        cat_freq_rows.append(
            {
                "Variable": col,
                "Category": "NaN" if is_missing else str(category),
                "Count": int(row["Count"]),
                "Percent_Pct": float(row["Percent (%)"]),
                "Failure_Rate_Pct": float(row["Failure_Rate (%)"]),
            }
        )

cat_freq_df = pd.DataFrame(cat_freq_rows)
cat_freq_df.to_csv(RESULTS_DIR / "eda_categorical_frequencies.csv", index=False)

# anomaly evidence in raw database (negative 1900-offset ages before cleaning)
raw_age_summary = raw_df["age"].describe().round(3).reset_index()
raw_age_summary.columns = ["Statistic", "Raw_Age_Value"]
raw_age_summary.to_csv(RESULTS_DIR / "eda_raw_age_anomaly.csv", index=False)
logger.info("Raw 'age' summary (with negative 1900 offsets):")
logger.info(raw_df["age"].describe().round(2).to_string())

## 3. Target variable distribution and Relevant attributes

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
target_counts = pre_train["target_risk"].value_counts().sort_index()
bars = ax.bar(
    ["0 — Aprovado", "1 — Risco de Reprovação"],
    target_counts.to_numpy(),
    color=PALETTE,
    edgecolor=COLOR_PURPLE_DARK,
)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        h + 120,
        f"{int(h)} ({h / len(pre_train) * 100:.1f}%)",
        ha="center",
        fontsize=9.5,
        fontweight="bold",
    )

ax.set_title(
    "Distribuição da Variável-Alvo (Treino Processado)",
    fontweight="bold",
    color=COLOR_PURPLE_DARK,
    pad=12,
)
ax.set_ylabel("Número de Estudantes", fontweight="bold")
ax.set_ylim(0, float(target_counts.max()) * 1.16)
ax.grid(axis="y", linestyle="--", alpha=0.8, color=COLOR_PURPLE_LIGHT)
ax.set_axisbelow(True)
sns.despine(top=True, right=True)
plt.tight_layout()

target_plot_path = IMAGES_DIR / f"eda_target_distribution_{RUN_TIMESTAMP}.png"
fig.savefig(target_plot_path, dpi=300, bbox_inches="tight")
logger.info(f"Saved plot: {target_plot_path}")
plt.show()

# numerical attributes distribution by risk class
plot_df = pre_train.copy()
plot_df["Situação"] = plot_df["target_risk"].map(
    {0: "0 — Aprovado", 1: "1 — Risco de Reprovação"}
)

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.8))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(
        data=plot_df,
        x=col,
        hue="Situação",
        hue_order=["0 — Aprovado", "1 — Risco de Reprovação"],
        kde=True,
        stat="density",
        common_norm=False,
        palette=PALETTE,
        ax=ax,
    )
    ax.set_title(
        f"Distribuição de {col} por Situação",
        fontweight="bold",
        color=COLOR_PURPLE_DARK,
    )
    ax.set_xlabel(col, fontweight="bold")
    ax.set_ylabel("Densidade")

sns.despine(top=True, right=True)
plt.tight_layout()

num_dist_path = IMAGES_DIR / f"eda_numerical_distributions_{RUN_TIMESTAMP}.png"
fig.savefig(num_dist_path, dpi=300, bbox_inches="tight")
logger.info(f"Saved plot: {num_dist_path}")
plt.show()

# 4. Missing Values Analysis

In [ ]:
missing_df = pd.DataFrame(
    {
        "Variable": pre_train.columns,
        "Missing_Count": pre_train.isna().sum().to_numpy(),
        "Missing_Pct": (pre_train.isna().mean() * 100).round(3).to_numpy(),
    }
)
missing_filtered = missing_df[missing_df["Missing_Count"] > 0].copy()
missing_filtered.to_csv(RESULTS_DIR / "eda_missing_values.csv", index=False)
display(missing_filtered)

# missing rate distribution across racial groups
missing_by_race_rows = []
for col in ["zfygpa", "fam_inc"]:
    if int(pre_train[col].isna().sum()) > 0:
        rates = (
            pre_train[col].isna().groupby(pre_train["race1"], dropna=False).mean() * 100
        ).round(2)
        logger.info(f"\nMissing rate of '{col}' by racial group (%):")
        logger.info(rates.to_string())
        
        for race_grp, rate_val in rates.items():
            is_missing = (
                race_grp is None
                or race_grp is pd.NA
                or (isinstance(race_grp, float) and np.isnan(race_grp))
            )
            missing_by_race_rows.append(
                {
                    "Variable": col,
                    "Race_Group": "NaN" if is_missing else str(race_grp),
                    "Missing_Rate_Pct": float(rate_val),
                }
            )

if missing_by_race_rows:
    pd.DataFrame(missing_by_race_rows).to_csv(
        RESULTS_DIR / "eda_missing_by_race.csv", index=False
    )

# 5. Correlation and Multicollinearity matrix

In [ ]:
corr_cols = ["lsat", "ugpa", "zfygpa", "age", "tier", "fulltime", "fam_inc", "target_risk"]
corr_matrix = pre_train[corr_cols].corr().round(4)
corr_matrix.to_csv(RESULTS_DIR / "eda_correlation_matrix.csv")

fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="Purples",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Correlação de Pearson"},
    ax=ax,
)

ax.set_title(
    "Matriz de Correlação (Treino Processado)",
    fontweight="bold",
    color=COLOR_PURPLE_DARK,
    pad=12,
)
plt.tight_layout()

corr_plot_path = IMAGES_DIR / f"eda_correlation_matrix_{RUN_TIMESTAMP}.png"
fig.savefig(corr_plot_path, dpi=300, bbox_inches="tight")
logger.info(f"Saved plot: {corr_plot_path}")
plt.show()

# 6. Outliers Detection

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 4.2))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=pre_train[col], color=COLOR_PURPLE_LIGHT, ax=ax)
    q1 = float(pre_train[col].quantile(0.25))
    q3 = float(pre_train[col].quantile(0.75))
    iqr = q3 - q1
    outliers = int(
        ((pre_train[col] < q1 - 1.5 * iqr) | (pre_train[col] > q3 + 1.5 * iqr)).sum()
    )
    ax.set_title(
        f"{col}\n(Outliers IIQ: {outliers})",
        fontweight="bold",
        color=COLOR_PURPLE_DARK,
    )
    ax.set_ylabel("Valor")

sns.despine(top=True, right=True)
plt.tight_layout()

outliers_plot_path = IMAGES_DIR / f"eda_outliers_boxplot_{RUN_TIMESTAMP}.png"
fig.savefig(outliers_plot_path, dpi=300, bbox_inches="tight")
logger.info(f"Saved plot: {outliers_plot_path}")
plt.show()

# 7. Hypothesys test (alpha = 0.05)

In [ ]:
ALPHA = 0.05
hypothesis_rows = []

# H1: zfygpa is lower among students with target_risk == 1 compared to target_risk == 0
z_pass = pre_train.loc[pre_train["target_risk"] == 0, "zfygpa"].dropna()
z_risk = pre_train.loc[pre_train["target_risk"] == 1, "zfygpa"].dropna()

welch_res: Any = stats.ttest_ind(z_pass, z_risk, equal_var=False, alternative="greater")
t_stat, p_welch = float(welch_res[0]), float(welch_res[1])

mwu_res: Any = stats.mannwhitneyu(z_pass, z_risk, alternative="greater")
u_stat, p_mwu = float(mwu_res[0]), float(mwu_res[1])

logger.info("=== H1: First-Year Performance (zfygpa) vs. Failure Risk ===")
logger.info(f"Mean/Median Passed (0): {float(z_pass.mean()):.3f} / {float(z_pass.median()):.3f}")
logger.info(f"Mean/Median Risk (1):   {float(z_risk.mean()):.3f} / {float(z_risk.median()):.3f}")
logger.info(f"Welch t-test: t = {t_stat:.4f}, p-value = {p_welch:.4e} -> Reject H0? {p_welch < ALPHA}")
logger.info(f"Mann-Whitney: U = {u_stat:.1f}, p-value = {p_mwu:.4e} -> Reject H0? {p_mwu < ALPHA}\n")

hypothesis_rows.extend(
    [
        {
            "Hypothesis": "H1 (zfygpa: Passed > At-Risk)",
            "Variable": "zfygpa",
            "Test": "Welch's t-test (one-sided)",
            "Statistic": round(t_stat, 4),
            "Degrees_of_Freedom": None,
            "p_value": p_welch,
            "Alpha": ALPHA,
            "Reject_H0": bool(p_welch < ALPHA),
        },
        {
            "Hypothesis": "H1 (zfygpa: Passed > At-Risk)",
            "Variable": "zfygpa",
            "Test": "Mann-Whitney U (one-sided)",
            "Statistic": round(u_stat, 2),
            "Degrees_of_Freedom": None,
            "p_value": p_mwu,
            "Alpha": ALPHA,
            "Reject_H0": bool(p_mwu < ALPHA),
        },
    ]
)

# H2: association between family income (fam_inc) / race (race1) and target_risk
for col in ["fam_inc", "race1"]:
    ct = pd.crosstab(pre_train[col], pre_train["target_risk"])
    chi2_res: Any = stats.chi2_contingency(ct)
    chi2, p_chi2, dof = float(chi2_res[0]), float(chi2_res[1]), int(chi2_res[2])

    risk_rate = (ct.loc[:, 1] / ct.sum(axis=1) * 100).round(2)
    logger.info(f"=== H2 ({col}): Categorical Association with target_risk ===")
    logger.info("Failure rate by category (%):")
    logger.info(risk_rate.to_string())
    logger.info(
        f"Chi-Square: chi2 = {chi2:.4f} (df={dof}), p-value = {p_chi2:.4e} -> Reject H0? {p_chi2 < ALPHA}\n"
    )

    hypothesis_rows.append(
        {
            "Hypothesis": f"H2 (Association: {col} vs target_risk)",
            "Variable": col,
            "Test": "Pearson's Chi-Square",
            "Statistic": round(chi2, 4),
            "Degrees_of_Freedom": dof,
            "p_value": p_chi2,
            "Alpha": ALPHA,
            "Reject_H0": bool(p_chi2 < ALPHA),
        }
    )

hyp_df = pd.DataFrame(hypothesis_rows)
hyp_df.to_csv(RESULTS_DIR / "eda_hypothesis_tests.csv", index=False)
display(hyp_df)

# 8. Comparison Before and After Pre-Process

In [ ]:
comp_df = pd.DataFrame(
    [
        {
            "Stage": "1. Raw Dataset (raw)",
            "Records": int(len(raw_df)),
            "Total_Columns": int(raw_df.shape[1]),
            "Missing_Values": int(raw_df.isna().sum().sum()),
            "Notes": "Original scales, negative 1900-offset ages, post-Year-1 leakage columns",
        },
        {
            "Stage": "2. Processed Split (pre_train)",
            "Records": int(len(pre_train)),
            "Total_Columns": int(pre_train.shape[1]),
            "Missing_Values": int(pre_train.isna().sum().sum()),
            "Notes": "65% stratified split, leakage removed, target_risk inverted, age fixed",
        },
        {
            "Stage": "3. Refined Train (train)",
            "Records": int(len(ref_train)),
            "Total_Columns": int(ref_train.shape[1]),
            "Missing_Values": int(ref_train.isna().sum().sum()),
            "Notes": "Train-fitted imputation, z-score scaling, one-hot cluster + 2 engineered features",
        },
    ]
)
comp_df.to_csv(RESULTS_DIR / "eda_preprocessing_comparison.csv", index=False)
display(comp_df)